# LorentzParT-JEPA: GSoC 2026 Demo

**Event Classification with Masked Transformer Autoencoders — JEPA Extension**

This notebook reproduces the three-way comparison:

| Condition | Pretraining | Fine-tune |
|-----------|------------|----------|
| JEPA      | Predicts target encoder *embeddings* of masked particles | Cross-entropy |
| MAE       | Reconstructs raw 4-vector features of masked particles | Cross-entropy |
| Scratch   | None | Cross-entropy from random init |

**Dataset**: 100k balanced subset of JetClass (10k per class, 80/10/10 split)

**Architecture**: LorentzParT — Particle Transformer with Lorentz-equivariant EquiLinear layers

---

### Key Idea: Why JEPA?

The existing MAE reconstructs low-level particle features (pT, η, φ, E). This forces the
encoder to memorise fine-grained detector noise rather than learning physics structure.

JEPA instead **predicts latent embeddings** — the target encoder's representation of the
masked particle — using an EMA-updated teacher. This trains the encoder on what the
jet *means* in learned feature space, not on pixel-level reconstruction.

The architecture borrows from Assran et al. (I-JEPA, CVPR 2023) but is adapted for
variable-length particle clouds with physics-inspired interaction embeddings.

## Setup

In [ ]:
import os, sys
# Run from the LorentzParT_JEPA directory
os.chdir(os.path.dirname(os.path.abspath('demo.ipynb')) if '__file__' not in dir() else os.path.dirname(__file__))
sys.path.insert(0, '.')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import torch

print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Step 1 — Prepare Data

Extract 100k balanced samples from the val_5M directory.
**Update `VAL_5M_DIR` to point to your copy of the val_5M ROOT files.**

In [ ]:
VAL_5M_DIR = '/path/to/val_5M'   # <-- UPDATE THIS
DATA_DIR   = './data'
SEED       = 42

# Skip if already prepared
if os.path.exists(os.path.join(DATA_DIR, 'train', 'particles.npy')):
    print('Data already prepared. Skipping.')
else:
    !python scripts/prepare_data.py --data-dir {VAL_5M_DIR} --output-dir {DATA_DIR} --seed {SEED}

In [ ]:
# Verify splits
for split in ['train', 'val', 'test']:
    p = np.load(os.path.join(DATA_DIR, split, 'particles.npy'))
    l = np.load(os.path.join(DATA_DIR, split, 'labels.npy'))
    print(f'{split:5s}: particles {p.shape}  labels {l.shape}')

# Class balance check
labels_train = np.load(os.path.join(DATA_DIR, 'train', 'labels.npy'))
class_counts = labels_train.sum(axis=0).astype(int)
class_names  = ['QCD/ZNuNu', 'H→bb', 'H→cc', 'H→gg', 'H→4q', 'H→ℓνqq', 'Z→qq', 'W→qq', 't→bqq', 't→bℓν']

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(class_names, class_counts)
ax.set_ylabel('Events')
ax.set_title('Training set class distribution (should be ~8,000 each)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## Step 2 — JEPA Pretraining

The context encoder sees a zeroed-out particle; the predictor must predict the
target encoder's embedding at that position. The target encoder is updated via EMA.

In [ ]:
!python scripts/pretrain_jepa.py \
    --data-dir {DATA_DIR} \
    --config-path configs/pretrain_jepa.yaml \
    --seed {SEED}

In [ ]:
import glob

def find_best(log_dir, model_class):
    files = sorted(
        glob.glob(os.path.join(log_dir, model_class, 'best', '*.pt')),
        key=os.path.getmtime, reverse=True
    )
    if not files: raise FileNotFoundError(f'No .pt found in {log_dir}/{model_class}/best/')
    return files[0]

jepa_weights = find_best('./logs', 'ParticleJEPA')
print(f'JEPA weights: {jepa_weights}')

## Step 3 — MAE Pretraining (Baseline)

Uses the existing LorentzParT MAE: reconstructs raw (pT, η, φ, E) via ConservationLoss.

In [ ]:
!python scripts/pretrain_mae.py \
    --data-dir {DATA_DIR} \
    --config-path configs/pretrain_mae.yaml \
    --seed {SEED}

In [ ]:
mae_weights = find_best('./logs', 'LorentzParT')
print(f'MAE weights: {mae_weights}')

## Step 4 — Fine-tuning (All Three Conditions)

In [ ]:
# JEPA pretrained → fine-tune
!python scripts/finetune.py \
    --data-dir {DATA_DIR} \
    --weights {jepa_weights} \
    --run-name jepa_finetune \
    --seed {SEED}

In [ ]:
# MAE pretrained → fine-tune
!python scripts/finetune.py \
    --data-dir {DATA_DIR} \
    --weights {mae_weights} \
    --run-name mae_finetune \
    --seed {SEED}

In [ ]:
# From scratch (no pretraining)
!python scripts/finetune.py \
    --data-dir {DATA_DIR} \
    --run-name scratch \
    --seed {SEED}

## Step 5 — Evaluation & Comparison

In [ ]:
import glob

# Helper to get most recent LorentzParT best model containing a name
def find_named(name):
    pattern = f'./logs/LorentzParT/best/{name}.pt'
    if os.path.exists(pattern):
        return pattern
    # Fall back to most recent if named not found
    files = sorted(
        glob.glob('./logs/LorentzParT/best/*.pt'),
        key=os.path.getmtime, reverse=True
    )
    return files[0] if files else None

jepa_ft  = find_named('jepa_finetune')
mae_ft   = find_named('mae_finetune')
scratch  = find_named('scratch')

print(f'JEPA fine-tuned: {jepa_ft}')
print(f'MAE fine-tuned:  {mae_ft}')
print(f'Scratch:         {scratch}')

In [ ]:
os.makedirs('./outputs', exist_ok=True)

for run_name, weights in [
    ('jepa_finetune', jepa_ft),
    ('mae_finetune',  mae_ft),
    ('scratch',       scratch),
]:
    if weights is None:
        print(f'Skipping {run_name}: no weights found')
        continue
    print(f'\n--- Evaluating: {run_name} ---')
    !python scripts/evaluate.py \
        --data-dir {DATA_DIR} \
        --weights {weights} \
        --run-name {run_name} \
        --outputs-dir ./outputs

## Step 6 — Visualise Results

In [ ]:
# Display ROC curves for all three conditions
from IPython.display import Image, display

for run_name in ['jepa_finetune', 'mae_finetune', 'scratch']:
    roc_path = f'./outputs/{run_name}_roc_curve.png'
    if os.path.exists(roc_path):
        print(f'\n=== {run_name} — ROC Curve ===')
        display(Image(roc_path))

In [ ]:
# Display confusion matrices
for run_name in ['jepa_finetune', 'mae_finetune', 'scratch']:
    cm_path = f'./outputs/{run_name}_confusion_matrix.png'
    if os.path.exists(cm_path):
        print(f'\n=== {run_name} — Confusion Matrix ===')
        display(Image(cm_path))

## Optional: Run Everything with One Command

Instead of running each cell, you can execute the full pipeline with:

```bash
python scripts/prepare_data.py --data-dir /path/to/val_5M --output-dir ./data
python scripts/run_comparison.py --data-dir ./data --seed 42
```

This runs all stages sequentially and prints a summary table.

---

## Architecture Summary

```
ParticleJEPA (JEPA pretraining)
├── processor        : (pT, η, φ, E) → 16-dim multivectors + pairwise U  
├── context_encoder  : LorentzParTEncoder on masked input  →  (B, 128, 128)
│     └── EquiLinear → Linear → 8× ParticleAttentionBlock
├── target_encoder   : EMA copy of context_encoder on full input  →  (B, 128, 128)
└── predictor        : bottleneck transformer (64-dim, 4 layers)
      input_proj → pos_embed → mask_token replace → 4× TransformerBlock → output_proj
      → predicted embedding  (B, 128)

Loss: MSELoss(LayerNorm(predicted), LayerNorm(target))
EMA momentum: 0.996 → 1.0 over training

Fine-tuning → LorentzParT (classification)
├── encoder          : loaded from context_encoder  (frozen or fine-tuned)
└── decoder          : CLS token + 2× ClassAttentionBlock + Linear(128→10)
```